# 00 — Data Download: Bhaduri et al. 2020

**Dataset:** "Cell stress in cortical organoids impairs molecular subtype specification"  
**Authors:** Bhaduri et al. 2020, Nature  
**Contents:** ~235k organoid cells (37 organoids, 3 protocols) + ~189k fetal cortex cells (GW 6–22)  
**Purpose:** Download raw count matrices from GEO to Google Drive for all downstream Colab notebooks.

**Run this notebook once.** After the data is on Drive, all other notebooks load from there.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Set Paths and Config

All paths are defined here. If you reorganize your Drive, only change this cell.

In [ ]:
import os

# --- Root folder on your Google Drive ---
DRIVE_ROOT = '/content/drive/MyDrive/brain-organoid-trajectories'

# --- Data directories ---
RAW_DIR      = os.path.join(DRIVE_ROOT, 'data', 'raw', 'bhaduri_2020')
PROCESSED_DIR = os.path.join(DRIVE_ROOT, 'data', 'processed', 'bhaduri_2020')

# --- GEO accession ---
# NOTE: Verify this accession at https://www.ncbi.nlm.nih.gov/geo/
# before running the download. Search for: Bhaduri 2020 cortical organoids stress
GEO_ACCESSION = 'GSE135827'  # <-- verify this before running

# Create directories
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print('Drive root:', DRIVE_ROOT)
print('Raw data dir:', RAW_DIR)
print('Processed dir:', PROCESSED_DIR)
print('GEO accession:', GEO_ACCESSION)

## 3. Verify GEO Accession — List Available Files

Before downloading, list all supplementary files for the accession.  
Confirm you see count matrix files (e.g. `*matrix*`, `*barcodes*`, `*features*` or `*.h5`).  
If the file list looks wrong, update `GEO_ACCESSION` in cell 2 and re-run.

In [ ]:
import ftplib

# Build FTP path from accession
# GEO FTP structure: /geo/series/GSEnnnnn/GSEaccession/suppl/
prefix = GEO_ACCESSION[:len(GEO_ACCESSION)-3] + 'nnn'  # e.g. GSE135nnn
ftp_path = f'/geo/series/{prefix}/{GEO_ACCESSION}/suppl/'

print(f'Checking FTP path: ftp.ncbi.nlm.nih.gov{ftp_path}\n')

ftp = ftplib.FTP('ftp.ncbi.nlm.nih.gov')
ftp.login()

try:
    files = ftp.nlst(ftp_path)
    print(f'Found {len(files)} file(s):')
    for f in files:
        print(' ', f.split('/')[-1])
except ftplib.error_perm as e:
    print(f'ERROR: {e}')
    print('The accession may be wrong. Check GEO_ACCESSION in cell 2.')

ftp.quit()

## 4. Download Supplementary Files

Downloads all files from the GEO supplementary folder to `RAW_DIR` on Drive.  
Skips files that already exist (safe to re-run).

In [ ]:
import ftplib
import os

def download_geo_suppl(accession, dest_dir):
    prefix = accession[:len(accession)-3] + 'nnn'
    ftp_path = f'/geo/series/{prefix}/{accession}/suppl/'

    ftp = ftplib.FTP('ftp.ncbi.nlm.nih.gov')
    ftp.login()

    files = ftp.nlst(ftp_path)
    print(f'Downloading {len(files)} file(s) to {dest_dir}\n')

    for filepath in files:
        filename = filepath.split('/')[-1]
        local_path = os.path.join(dest_dir, filename)

        if os.path.exists(local_path):
            size_mb = os.path.getsize(local_path) / 1e6
            print(f'  SKIP (exists, {size_mb:.1f} MB): {filename}')
            continue

        print(f'  Downloading: {filename} ...', end=' ', flush=True)
        with open(local_path, 'wb') as f:
            ftp.retrbinary(f'RETR {filepath}', f.write)
        size_mb = os.path.getsize(local_path) / 1e6
        print(f'done ({size_mb:.1f} MB)')

    ftp.quit()
    print('\nAll downloads complete.')

download_geo_suppl(GEO_ACCESSION, RAW_DIR)

## 5. Verify Download — List Files on Drive

In [ ]:
print(f'Contents of {RAW_DIR}:\n')
total_mb = 0
for fname in sorted(os.listdir(RAW_DIR)):
    fpath = os.path.join(RAW_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1e6
    total_mb += size_mb
    print(f'  {fname:<60} {size_mb:>8.1f} MB')
print(f'\n  Total: {total_mb:.1f} MB')

## 6. Sanity Check — Peek at the Data

Load the data and print basic stats to confirm it looks right before committing to full preprocessing.  
Adjust the filename below to match whatever was downloaded.

In [ ]:
!pip install -q scanpy

In [ ]:
import scanpy as sc

# --- Update this to the actual filename after checking cell 5 output ---
# Common patterns from GEO: *_matrix.txt.gz, *_counts.h5, *_raw_counts.h5ad
# List files to find the right one:
files = sorted(os.listdir(RAW_DIR))
for i, f in enumerate(files):
    print(f'[{i}] {f}')

In [ ]:
# Set this to the index of the main count matrix file from the list above
FILE_INDEX = 0
data_file = os.path.join(RAW_DIR, files[FILE_INDEX])
print(f'Loading: {data_file}')

# Load depending on format
if data_file.endswith('.h5ad'):
    adata = sc.read_h5ad(data_file)
elif data_file.endswith('.h5'):
    adata = sc.read_10x_h5(data_file)
elif data_file.endswith('.txt.gz') or data_file.endswith('.txt'):
    adata = sc.read_text(data_file).T  # GEO text matrices are genes x cells, scanpy wants cells x genes
else:
    print('Unrecognized format — check the file and load manually.')
    adata = None

if adata is not None:
    print(f'\nShape: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes')
    print(f'obs columns: {list(adata.obs.columns)}')
    print(f'var columns: {list(adata.var.columns)}')
    print('\nFirst 5 cell barcodes:', list(adata.obs_names[:5]))
    print('First 5 genes:', list(adata.var_names[:5]))

## 7. Save Verified AnnData to Processed Dir

Save the raw (unmodified) loaded object as `.h5ad` for fast loading in downstream notebooks.  
This is just format conversion — no filtering or normalization yet.

In [ ]:
if adata is not None:
    out_path = os.path.join(PROCESSED_DIR, 'bhaduri_2020_raw.h5ad')
    adata.write_h5ad(out_path)
    size_mb = os.path.getsize(out_path) / 1e6
    print(f'Saved to: {out_path} ({size_mb:.1f} MB)')
else:
    print('No adata to save — fix loading in cell 6 first.')

## Done

Data is now on Google Drive at:
- **Raw files:** `brain-organoid-trajectories/data/raw/bhaduri_2020/`
- **h5ad copy:** `brain-organoid-trajectories/data/processed/bhaduri_2020/bhaduri_2020_raw.h5ad`

Next notebook: `colab_01_preprocessing.ipynb` — QC, filtering, normalization, HVG, PCA on the full dataset.